# Лабортаторная работа № 2 "Статистический анализ одномерных выборок. Проверка статистических гипотез"

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from IPython.display import display

In [2]:
file_name = "data1.xlsx"

df = pd.read_excel(
    file_name,
    sheet_name=1,
    usecols="A:G",
    header = None,
    engine="openpyxl"
)

samples = [df[col].dropna().values for col in df.columns]
print(df.head())

           0          1          2          3   4   5          6
0 -16.427992  10.337568  10.094972  12.235925  10  16  -2.336080
1 -29.368694  20.601542  15.573639  12.159321   6  15   5.041232
2  -6.561724   1.876631   2.625150  24.552468  10  14   6.987623
3   7.358867  17.243398  15.855155  12.348476   9  14   9.542421
4   6.692038 -23.179041  11.742064  19.597243   9  15  11.116361


In [3]:
results = []
confidence = 0.95

for col in df.columns:
    data = df[col].dropna()
    n = len(data)

    mean = data.mean()
    median = data.median()
    mode = data.mode().iloc[0] if not data.mode().empty else np.nan
    std = data.std(ddof=1)
    var = data.var(ddof=1)
    sem = stats.sem(data)
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)
    min_val = data.min()
    max_val = data.max()
    data_range = max_val - min_val
    total_sum = data.sum()

    t_crit = stats.t.ppf((1 + confidence) / 2, df=n-1)
    margin = t_crit * sem
    ci_lower = mean - margin
    ci_upper = mean + margin

    results.append([
        n, mean, sem, median, mode, std, var,
        kurt, skew, data_range, min_val, max_val,
        total_sum, ci_lower, ci_upper
    ])

columns = [
    "N", "Среднее", "Стд. ошибка", "Медиана", "Мода",
    "Стд. откл.", "Дисперсия", "Эксцесс", "Асимметрия",
    "Размах", "Минимум", "Максимум", "Сумма",
    "CI 95% (ниж)", "CI 95% (верх)"
]

stats_table = pd.DataFrame(results, columns=columns, index=df.columns).style

stats_table

,N,Среднее,Стд. ошибка,Медиана,Мода,Стд. откл.,Дисперсия,Эксцесс,Асимметрия,Размах,Минимум,Максимум,Сумма,CI 95% (ниж),CI 95% (верх)
0,500,-10.356208,0.607520,-9.587024,-20.190313,13.584549,184.539975,-1.246367,-0.071625,45.765557,-33.870846,11.894711,-5178.103885,-11.549819,-9.162596
1,500,8.813767,0.819621,8.710024,-23.627036,18.327286,335.889412,0.287275,-0.038906,112.458074,-49.504520,62.953554,4406.883337,7.203433,10.424100
2,500,8.815445,0.599740,8.947190,8.427380,13.410601,179.844218,-0.229775,-0.043017,80.036116,-28.674916,51.361200,4407.722347,7.637117,9.993772
3,500,12.626233,0.312670,12.756511,14.506429,6.991504,48.881130,0.027581,0.152056,42.804531,-7.461513,35.343018,6313.116390,12.011922,13.240544
4,500,8.392000,0.072769,8.000000,9.000000,1.627154,2.647631,-0.343639,-0.188030,9.000000,3.000000,12.000000,4196.000000,8.249029,8.534971
5,500,12.114000,0.150650,12.000000,12.000000,3.368635,11.347699,0.354309,0.392514,23.000000,3.000000,26.000000,6057.000000,11.818014,12.409986
6,500,7.921503,0.171443,7.737115,6.583730,3.833575,14.696298,0.301399,0.016765,26.375637,-5.948884,20.426754,3960.751652,7.584665,8.258342


### Гипотезы:
**Выборка 0** - равномерное распределение  
**Выборка 1** - нормальное распределение  
**Выборка 2** - нормальное распределение  
**Выборка 3** - нормальное распределение  
**Выборка 4** - биноминальное распределение  
**Выборка 5** - пуассоновское распределение  
**Выборка 6** - нормальное распределение  

### χ² равномерное
$$\chi^2 = \sum_{i=1}^{k} \frac{\left(n_i - \frac{n}{k}\right)^2}{\frac{n}{k}}$$
$$E_i = \frac{n}{k}$$

In [4]:
def chi_square_uniform(sample, bins=10):
    hist, bin_edges = np.histogram(sample, bins=bins)
    expected = np.full_like(hist, len(sample) / bins)
    chi2, p = stats.chisquare(hist, expected)
    return p

### χ² нормальное
$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$
$$E_i = n \cdot \left[ F(x_i; \mu, \sigma) - F(x_{i-1}; \mu, \sigma) \right]$$

In [5]:
def chi_square_norm(sample):
    n = len(sample)
    k = int(1 + 3.322 * np.log10(n))
    bins = np.histogram_bin_edges(sample, bins=k)
    observed, _ = np.histogram(sample, bins=bins)
    mu, sigma = np.mean(sample), np.std(sample, ddof=1)
    cdf_vals = stats.norm.cdf(bins, mu, sigma)
    expected = n * np.diff(cdf_vals)
    mask = expected > 5
    observed = observed[mask]
    expected = expected[mask]
    if len(observed) < 2:
        return np.nan
    
    chi2 = np.sum((observed - expected)**2 / expected)
    dfree = len(observed) - 1 - 2
    return 1 - stats.chi2.cdf(chi2, dfree)

### χ² Пуассоновское
$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$
$$E_i = n \cdot \frac{\lambda^k e^{-\lambda}}{k!}$$

In [6]:
def chi_square_poisson(sample):
    values, counts = np.unique(sample, return_counts=True)
    n = len(sample)
    lam = np.mean(sample)
    probs = stats.poisson.pmf(values, lam)
    expected = n * probs
    mask = expected > 5
    observed = counts[mask]
    expected = expected[mask]
    if len(observed) < 2:
        return np.nan
    
    chi2 = np.sum((observed - expected)**2 / expected)
    dfree = len(observed) - 1 - 1
    return 1 - stats.chi2.cdf(chi2, dfree)

### χ² биномиальное
$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$
$$E_i = N \cdot \binom{n}{k} p^k (1 - p)^{n - k}$$

In [7]:
def chi_square_binomial(sample):
    values, counts = np.unique(sample, return_counts=True)
    N = len(sample)
    mean = np.mean(sample)
    var = np.var(sample, ddof=1)
    if mean <= var:
        return np.nan
    
    n_est = int(round(mean**2 / (mean - var)))
    if n_est <= 0:
        return np.nan
    
    p_est = mean / n_est
    probs = stats.binom.pmf(values, n_est, p_est)
    expected = N * probs
    mask = expected > 5
    observed = counts[mask]
    expected = expected[mask]
    if len(observed) < 2:
        return np.nan
    
    chi2 = np.sum((observed - expected)**2 / expected)
    dfree = len(observed) - 1 - 2
    
    return 1 - stats.chi2.cdf(chi2, dfree)


### Критерий Колмогорова–Смирнова
$$D = \sup_x \left| F_n(x) - F(x) \right|$$
$$F_n(x) = \frac{1}{n} \sum_{i=1}^{n} I(x_i \le x)$$

In [8]:
results = []

for i, sample in enumerate(samples):
    is_discrete = len(np.unique(sample)) < 20

    if not is_discrete:
        mu, sigma = np.mean(sample), np.std(sample, ddof=1)
        _, ks_p = stats.kstest(sample, 'norm', args=(mu, sigma))

        a, b = np.min(sample), np.max(sample)
        _, ks_uniform_p = stats.kstest(sample, 'uniform', args=(a, b - a))
    else:
        ks_p = np.nan
        ks_uniform_p = np.nan

    chi2_norm_p = chi_square_norm(sample)
    chi2_pois_p = chi_square_poisson(sample)
    chi2_binom_p = chi_square_binomial(sample)
    chi2_uniform_p = chi_square_uniform(sample)

    if not is_discrete:
        if ks_p > 0.05 and chi2_norm_p > 0.05:
            conclusion = "Нормальное"
        elif ks_uniform_p > 0.05 and chi2_uniform_p > 0.05:
            conclusion = "Равномерное"
        else:
            conclusion = "Не нормальное"
    else:
        if chi2_pois_p > 0.05:
            conclusion = "Пуассон"
        elif chi2_binom_p > 0.05:
            conclusion = "Биномиальное"
        else:
            conclusion = "Другое дискретное"

    results.append({
        "sample": i,
        "KS_norm_p": ks_p,
        "KS_uniform_p": ks_uniform_p,
        "Chi2_norm_p": chi2_norm_p,
        "Chi2_uniform_p": chi2_uniform_p,
        "Chi2_pois_p": chi2_pois_p,
        "Chi2_binom_p": chi2_binom_p,
        "Conclusion": conclusion
    })

results_df = pd.DataFrame(results)
print(results_df)

   sample  KS_norm_p  KS_uniform_p   Chi2_norm_p  Chi2_uniform_p  Chi2_pois_p  \
0       0   0.007100  2.716948e-01  0.000000e+00    1.699807e-01          NaN   
1       1   0.935086  1.350698e-22  5.421934e-01    1.448393e-75          NaN   
2       2   0.555121  1.959411e-22  5.102743e-01    1.196007e-63          NaN   
3       3   0.970282  1.292590e-24  2.624477e-01    1.276998e-71          NaN   
4       4        NaN           NaN  2.220446e-16    2.215603e-69     0.000000   
5       5   0.001253  1.066517e-49  1.112859e-03   1.830855e-114     0.895872   
6       6   0.795283  2.241279e-27  5.928333e-01    9.412889e-97          NaN   

   Chi2_binom_p         Conclusion  
0           NaN        Равномерное  
1           NaN         Нормальное  
2           NaN         Нормальное  
3           NaN         Нормальное  
4      0.030949  Другое дискретное  
5      0.827298      Не нормальное  
6           NaN         Нормальное  


**Выборка 1**
Равномерное распределение

**Выборки 2, 3 и 6**
p-value по К-С и χ² для нормального распределения > 0.05
нет оснований отвергать гипотезу нормальности

**Выборки 0 и 5**
p-value < 0.05
гипотеза нормальности отвергается

**Выборка 4**
норальное распределение не подходит (p ≈ 0)
для Пуассона гипотеза отвергается
биномиальное — частично проходит, но в целом нестабильно
дискретная, но не соответствует ни одному из проверенных стандартных распределений

In [16]:
m_cols = [0, 1, 2, 3]
for col in m_cols:
    sample = df[col].values
    true_mean = np.mean(sample)
    rounded_mean = int(np.round(true_mean))
    n = len(sample)
    df_t = n - 1

    t_stat, p_value = stats.ttest_1samp(sample, rounded_mean)

    alpha = 0.05
    t_critical = stats.t.ppf(1 - alpha/2, df_t)

In [17]:
import numpy as np
import pandas as pd
from scipy import stats

results = []

for i, sample in enumerate(samples):
    x_bar = np.mean(sample)
    mu_rounded = np.round(x_bar)

    s = np.std(sample, ddof=1)
    n = len(sample)

    if s == 0:
        t_stat = np.nan
        p_value = 1.0
        conclusion = "Ок"
    else:
        t_stat = (x_bar - mu_rounded) / (s / np.sqrt(n))
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))

        if p_value > 0.05:
            conclusion = "Можно округлять"
        else:
            conclusion = "Нельзя округлять"

    results.append({
        "sample": i,
        "mean": x_bar,
        "rounded_mean": mu_rounded,
        "t_stat": t_stat,
        "p_value": p_value,
        "conclusion": conclusion
    })

df = pd.DataFrame(results)
display(df)

,sample,mean,rounded_mean,t_stat,p_value,conclusion
0,0,-10.356208,-10.0,-0.586331,5.579180e-01,Можно округлять
1,1,8.813767,9.0,-0.227219,8.203467e-01,Можно округлять
2,2,8.815445,9.0,-0.307725,7.584197e-01,Можно округлять
3,3,12.626233,13.0,-1.195406,2.324961e-01,Можно округлять
4,4,8.392000,8.0,5.386942,1.106219e-07,Нельзя округлять
5,5,12.114000,12.0,0.756721,4.495740e-01,Можно округлять
6,6,7.921503,8.0,-0.457860,6.472526e-01,Можно округлять


### Критерий Фишера (равенство дисперсий)
$$F = \frac{s_1^2}{s_2^2}$$

### t-Критерий Стьюдента (равные дисперсии)
$$t = \frac{\bar{X}_1 - \bar{X}_2}{s_p \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}}, \quad$$
$$s_p^2 = \frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}$$

### t-Критерий Уэлча (неравные дисперсии)
$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}, \quad$$
$$\nu = \frac{\left( \frac{s_1^2}{n_1} + \frac{s_2^2}{n_2} \right)^2}{\frac{(s_1^2/n_1)^2}{n_1 - 1} + \frac{(s_2^2/n_2)^2}{n_2 - 1}}$$

In [9]:
def compare(x1, x2, alpha=0.05):
    x1 = np.array(x1)
    x2 = np.array(x2)

    s1 = np.var(x1, ddof=1)
    s2 = np.var(x2, ddof=1)

    if s1 >= s2:
        F = s1 / s2
        dfn = len(x1) - 1
        dfd = len(x2) - 1
    else:
        F = s2 / s1
        dfn = len(x2) - 1
        dfd = len(x1) - 1

    p_f = 2 * min(stats.f.cdf(F, dfn, dfd), 1 - stats.f.cdf(F, dfn, dfd))

    equal_var = p_f > alpha

    t_stat, p_t = stats.ttest_ind(x1, x2, equal_var=equal_var)

    return F, p_f, t_stat, p_t, equal_var, p_t > alpha


samples_idx = [1, 2, 3, 6]
results = []

for i in range(len(samples_idx)):
    for j in range(i + 1, len(samples_idx)):
        s1 = df.iloc[:, samples_idx[i]].dropna()
        s2 = df.iloc[:, samples_idx[j]].dropna()

        F, p_f, t_stat, p_t, eq_var, eq_mean = compare(s1, s2)

        results.append({
            "pair": f"{samples_idx[i]}-{samples_idx[j]}",
            "F_p": p_f,
            "t_p": p_t,
            "equal_variances": eq_var,
            "equal_means": eq_mean
        })

results_df = pd.DataFrame(results)
display(results_df)

,pair,F_p,t_p,equal_variances,equal_means
0,1-2,4.570344e-12,9.986821e-01,False,True
1,1-3,2.220446e-16,1.611832e-05,False,False
2,1-6,2.220446e-16,2.870931e-01,False,True
3,2-3,2.220446e-16,2.485060e-08,False,False
4,2-6,2.220446e-16,1.523562e-01,False,True
5,3-6,2.220446e-16,5.332194e-36,False,False


По результатам F-критерия установлено, что дисперсии всех пар выборок различаются.
С использованием t-критерия Уэлча установлено, что математические ожидания выборок 1, 2 и 6 статистически не различаются, в то время как выборка 3 имеет значимо отличающееся математическое ожидание от всех остальных.

### Критерий Смирнова-Колмогорова
$$D = \sup_x \left| F_1(x) - F_2(x) \right|$$

### Критерий Манна-Уитни
$$U = n_1 n_2 + \frac{n_1 (n_1 + 1)}{2} - R_1$$

### Критерий Уилкоксона
$$W = \sum_{i=1}^{n} \operatorname{sign}(d_i) \cdot R_i$$

In [10]:
samples_idx = [1, 2, 3, 6]
results = []
for i in range(len(samples_idx)):
    for j in range(i + 1, len(samples_idx)):
        x1 = df.iloc[:, samples_idx[i]].dropna()
        x2 = df.iloc[:, samples_idx[j]].dropna()

        ks_stat, ks_p = stats.ks_2samp(x1, x2)
        mw_stat, mw_p = stats.mannwhitneyu(x1, x2, alternative='two-sided')
        w_stat, w_p = stats.wilcoxon(x1[:min(len(x1), len(x2))], x2[:min(len(x1), len(x2))])

        results.append({
            "pair": f"{samples_idx[i]}-{samples_idx[j]}",
            "KS_p": ks_p,
            "MW_p": mw_p,
            "Wilcoxon_p": w_p,
            "KS_same_dist": ks_p > 0.05,
            "MW_same_median": mw_p > 0.05,
            "Wilcoxon_same": w_p > 0.05
        })

results_df = pd.DataFrame(results)
display(results_df)

,pair,KS_p,MW_p,Wilcoxon_p,KS_same_dist,MW_same_median,Wilcoxon_same
0,1-2,8.150167e-02,9.780753e-01,9.943226e-01,True,True,True
1,1-3,4.568072e-21,9.306296e-06,4.473064e-05,False,False,False
2,1-6,8.535314e-25,2.980146e-01,2.299943e-01,False,True,True
3,2-3,5.833973e-16,4.886234e-07,5.082585e-08,False,False,False
4,2-6,1.297975e-23,1.053947e-01,1.361565e-01,False,True,True
5,3-6,1.309317e-37,4.379789e-33,5.744238e-31,False,False,False


По результатам критериев Колмогорова–Смирнова, Манна–Уитни и Уилкоксона установлено, что выборки 1, 2 и 6 являются однородными по центральным характеристикам.
Выборка 3 статистически значимо отличается от остальных.
Для некоторых пар наблюдается различие распределений по форме (критерий KS), однако медианы остаются статистически неразличимыми.

### Критерий Манна-Кендалла
$$p = 2 \left(1 - \Phi\left(|Z|\right)\right)$$
$$S = \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \operatorname{sign}(x_j - x_i)$$

In [13]:
def mann_kendall(x):
    n = len(x)
    S = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            S += np.sign(x[j] - x[i])
    var_S = (n * (n - 1) * (2*n + 5)) / 18
    if S > 0:
        Z = (S - 1) / np.sqrt(var_S)
    elif S < 0:
        Z = (S + 1) / np.sqrt(var_S)
    else:
        Z = 0
    p = 2 * (1 - stats.norm.cdf(abs(Z)))
    return S, Z, p

In [15]:
results = []
for i in range(df.shape[1]):
    x = df.iloc[:, i].dropna().values
    t = np.arange(len(x))
    S, Z, p_mk = mann_kendall(x)
    slope, intercept, r_value, p_reg, std_err = stats.linregress(t, x)
    results.append({
        "sample": i,
        "MK_p": p_mk,
        "MK_trend": "есть" if p_mk < 0.05 else "нет",
        "MK_direction": "возрастает" if S > 0 else "убывает" if S < 0 else "нет",
        "Reg_p": p_reg,
        "Reg_trend": "есть" if p_reg < 0.05 else "нет",
        "slope": slope
    })
results_df = pd.DataFrame(results)
results_df

,sample,MK_p,MK_trend,MK_direction,Reg_p,Reg_trend,slope
0,0,0.784839,нет,убывает,0.823632,нет,-0.000939
1,1,0.833206,нет,убывает,0.899021,нет,-0.000722
2,2,0.787724,нет,возрастает,0.657553,нет,0.001845
3,3,0.077276,нет,убывает,0.045963,есть,-0.004321
4,4,0.617863,нет,убывает,0.569508,нет,-0.000287
5,5,0.478197,нет,убывает,0.320643,нет,-0.001038
6,6,0.908917,нет,убывает,0.980932,нет,-0.000028
